# Notebook 41 — Grokking Forward-Only (nb39 v2)

Re-do of nb39 with two corrections:
1. **Qwen3.6 LoRA key fix** (`feedback_qwen36_lora_language_model_key_mismatch.md`): strip `.language_model.` from saved adapter keys before `PeftModel.from_pretrained()`
2. **Forward-only** (no generation): feed `prompt + chosen` from nb37 pairs.json through each checkpoint, capture L31/L55 at deterministic end-of-think position. Same scientific signal as nb39 generation loop, 10× cheaper.

**Hypothesis**: probe AUROC for preference-shifted output undergoes phase transition during DPO training (detectable via activations even when greedy decode is identical between base and LoRA).

**Pre-confirmed (nb40)**: DPO has real effect (logit max diff 0.654 base vs LoRA), but argmax tokens identical at greedy. Effect is sub-greedy-threshold. Activations at L31/L55 still differ because LoRA modifies residual stream at every layer.

**Drive**: `/content/drive/MyDrive/openinterp_runs/41_grokking_forward_only/`
**Compute**: ~10 min on RTX 6000.


## Phase 1 — Setup


In [ ]:
from pathlib import Path
import os, json, time, shutil
import torch, numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / '41_grokking_forward_only'
OUT.mkdir(parents=True, exist_ok=True)

NB37 = DRIVE / 'openinterp_runs' / '37_multiprobe_dpo_full'
DPO_RUN = NB37 / 'dpo_run'
print(f'OUT: {OUT}')
print(f'nb37 dpo_run: {DPO_RUN.exists()}')

!pip install -q -U torchao
!pip install -q -U transformers accelerate peft huggingface_hub safetensors scikit-learn


## Phase 2 — Load Qwen3.6-27B base + probes + HF


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, hf_hub_download, HfApi, create_repo
import getpass, joblib

CFG = {
    'model_id':         'Qwen/Qwen3.6-27B',
    'capture_layer_fg': 31,
    'capture_layer_rg': 55,
    'n_holdout':        20,
    'random_seed':      42,
    'fg_probe_repo':    'caiovicentino1/FabricationGuard-linearprobe-qwen36-27b',
    'rg_probe_repo':    'caiovicentino1/ReasoningGuard-linearprobe-qwen36-27b',
    'output_repo':      'caiovicentino1/openinterp-41-grokking-forward-only',
    'gate_ratio':       2.0,
}
THINK_CLOSE_ID = 248069
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda'; assert torch.cuda.is_available()
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
base_model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto',
)
base_model.eval()
print(f'✓ Base loaded — {torch.cuda.get_device_name(0)}, layers={len(base_model.model.layers)}')

# Probes
fg_path = hf_hub_download(repo_id=CFG['fg_probe_repo'], filename='probe.joblib', repo_type='dataset')
rg_path = hf_hub_download(repo_id=CFG['rg_probe_repo'], filename='probe.joblib', repo_type='dataset')
fg_artifact = joblib.load(fg_path)
rg_artifact = joblib.load(rg_path)
fg_clf = fg_artifact['probe']; fg_scaler = fg_artifact['scaler']
rg_clf = rg_artifact['probe']; rg_scaler = rg_artifact['scaler']
print(f'✓ FG: {type(fg_clf).__name__}, RG: {type(rg_clf).__name__}')

def fg_score(activation):
    x = activation.float().cpu().numpy().reshape(1, -1)
    return float(fg_clf.predict_proba(fg_scaler.transform(x))[0, 1])
def rg_score(activation):
    x = activation.float().cpu().numpy().reshape(1, -1)
    return float(rg_clf.predict_proba(rg_scaler.transform(x))[0, 1])


## Phase 3 — Fix + discover checkpoints

Apply Qwen3.6 LoRA key fix (strip `.language_model.`) to all available checkpoints.


In [ ]:
from safetensors.torch import load_file, save_file
from peft import PeftModel

def fix_qwen36_adapter_keys(adapter_dir, fixed_dir):
    """Strip .language_model. from saved Qwen3.6 LoRA keys for dense reload."""
    src = adapter_dir / 'adapter_model.safetensors'
    state = load_file(str(src))
    fixed = {k.replace('.language_model.', '.'): v for k, v in state.items()}
    fixed_dir.mkdir(parents=True, exist_ok=True)
    save_file(fixed, str(fixed_dir / 'adapter_model.safetensors'))
    shutil.copy(adapter_dir / 'adapter_config.json', fixed_dir / 'adapter_config.json')
    return fixed_dir

# Discover available checkpoints in nb37
checkpoints = []
if DPO_RUN.exists():
    for d in sorted(DPO_RUN.iterdir()):
        if d.is_dir() and d.name.startswith('checkpoint-'):
            adapter = d / 'adapter_model.safetensors'
            if adapter.exists():
                step = int(d.name.split('-')[1])
                checkpoints.append({'step': step, 'orig_dir': d})
checkpoints.sort(key=lambda x: x['step'])

# Apply fix to each
fixed_root = OUT / 'fixed_adapters'
for c in checkpoints:
    fdir = fixed_root / f"step-{c['step']}"
    fix_qwen36_adapter_keys(c['orig_dir'], fdir)
    c['fixed_dir'] = fdir
    print(f"  step {c['step']}: fixed → {fdir}")
print(f'✓ {len(checkpoints)} checkpoints prepared')


## Phase 4 — Hooks + 20 hold-out prompts


In [ ]:
captured = {}
_pos = {'pos': None}

def make_hook(layer_idx):
    def hook(module, input, output):
        h = output[0] if isinstance(output, tuple) else output
        pos = _pos['pos']
        if pos is None or pos >= h.shape[1]:
            return
        captured[f'L{layer_idx}'] = h[0, pos, :].detach().cpu().to(torch.float16).clone()
    return hook

def get_layers(m):
    """Layer access for Qwen3.6 dense or PEFT-wrapped."""
    candidates = [
        lambda x: x.base_model.model.model.layers,
        lambda x: x.model.layers,
    ]
    for fn in candidates:
        try:
            l = fn(m)
            if hasattr(l, '__len__') and len(l) > 0: return l
        except AttributeError: continue
    raise RuntimeError(f'Cannot find layers in {type(m).__name__}')

# Holdout prompts from nb37 pairs.json
with open(NB37 / 'pairs.json') as f:
    all_pairs = json.load(f)
rng = np.random.default_rng(CFG['random_seed'])
indices = rng.choice(len(all_pairs), size=CFG['n_holdout'], replace=False)
holdout = [all_pairs[i] for i in indices]
print(f'Hold-out: {len(holdout)} prompts')
src_dist = {}
for p in holdout: src_dist[p['src']] = src_dist.get(p['src'], 0) + 1
print(f'Sources: {src_dist}')


## Phase 5 — Forward-only score per checkpoint

For each checkpoint, build (prompt + chosen) text, tokenize, forward pass, capture L31/L55 at end-of-think token. No generation.


In [ ]:
from tqdm.auto import tqdm
import gc

def find_end_think_pos(token_ids):
    ids = token_ids.tolist() if hasattr(token_ids, 'tolist') else list(token_ids)
    for i in range(len(ids) - 1, -1, -1):
        if ids[i] == THINK_CLOSE_ID:
            return i
    return None

def score_one(model, prompt, chosen):
    """Forward pass on prompt+chosen, return (fg, rg, end_pos, act_fg)."""
    # Build full text: chat-templated prompt then assistant response with </think> intact
    messages_input = [{'role': 'user', 'content': prompt}]
    user_text = tok.apply_chat_template(messages_input, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    full_text = user_text + chosen + '<|im_end|>'
    enc = tok(full_text, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    end_pos = find_end_think_pos(ids[0])
    if end_pos is None:
        return {'fg': float('nan'), 'rg': float('nan'), 'end_pos': None}
    captured.clear()
    _pos['pos'] = end_pos
    with torch.no_grad():
        _ = model(ids)
    act_fg = captured.get(f'L{CFG["capture_layer_fg"]}')
    act_rg = captured.get(f'L{CFG["capture_layer_rg"]}')
    return {
        'fg': fg_score(act_fg) if act_fg is not None else float('nan'),
        'rg': rg_score(act_rg) if act_rg is not None else float('nan'),
        'end_pos': end_pos,
        'act_fg': act_fg.clone() if act_fg is not None else None,
    }

# Build sequence: base first (no adapter)
ckpt_seq = [{'step': 0, 'fixed_dir': None, 'name': 'base'}] + checkpoints
print(f'Will score {len(ckpt_seq)} checkpoints × {CFG["n_holdout"]} prompts = {len(ckpt_seq) * CFG["n_holdout"]} forward passes')


In [ ]:
results_path = OUT / 'forward_only_results.jsonl'
acts_dir = OUT / 'acts'
acts_dir.mkdir(exist_ok=True)

# Resume support
done_keys = set()
if results_path.exists():
    with open(results_path) as f:
        for line in f:
            try:
                r = json.loads(line)
                done_keys.add(f"step{r['step']}_{r['pair_id']}")
            except: continue
    print(f'Resume: {len(done_keys)} done')

# Hooks register on base initially
hook_handles = []
for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
    h = get_layers(base_model)[L].register_forward_hook(make_hook(L))
    hook_handles.append(h)

current_model = base_model

for ck_i, ck in enumerate(ckpt_seq):
    print(f'\n=== Checkpoint {ck_i+1}/{len(ckpt_seq)}: step={ck["step"]} ===')
    if ck['step'] == 0:
        # Use base (already loaded with hooks)
        active = base_model
    else:
        # Detach previous adapter if any
        for h in hook_handles:
            try: h.remove()
            except: pass
        hook_handles = []
        # Load LoRA via canonical API on the base model (NOT on previous lora_model)
        try:
            active = PeftModel.from_pretrained(base_model, str(ck['fixed_dir']), is_trainable=False)
            active.eval()
            print(f'  ✓ Loaded LoRA step {ck["step"]}')
        except Exception as e:
            print(f'  ❌ Load failed: {e}')
            continue
        # Re-register hooks on the wrapped model
        for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
            h = get_layers(active)[L].register_forward_hook(make_hook(L))
            hook_handles.append(h)
    
    for p in tqdm(holdout, desc=f'step{ck["step"]}'):
        key = f"step{ck['step']}_{p['id']}"
        if key in done_keys: continue
        try:
            res = score_one(active, p['prompt'], p['chosen'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f'OOM on {key}'); continue
        # Save activation
        if res.get('act_fg') is not None:
            torch.save(res['act_fg'], acts_dir / f'{key}.pt')
        record = {
            'step': ck['step'], 'pair_id': p['id'], 'src': p['src'],
            'fg': res['fg'], 'rg': res['rg'], 'end_pos': res['end_pos'],
        }
        with open(results_path, 'a') as f:
            f.write(json.dumps(record) + '\n')
        done_keys.add(key)
    
    # Unload LoRA back to base for next iteration (avoid stacking adapters)
    if ck['step'] != 0 and isinstance(active, PeftModel):
        # Re-attach hooks to base model for the next non-zero checkpoint
        for h in hook_handles:
            try: h.remove()
            except: pass
        hook_handles = []
        # Drop reference to lora_model; base still loaded
        del active
        torch.cuda.empty_cache(); gc.collect()
        for L in [CFG['capture_layer_fg'], CFG['capture_layer_rg']]:
            h = get_layers(base_model)[L].register_forward_hook(make_hook(L))
            hook_handles.append(h)

print('\n✓ Phase 5 complete')
(OUT / '_phase5_done.txt').write_text(f'ts={time.time()}')


## Phase 6 — Analysis: phase transition or gradual?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

with open(results_path) as f:
    records = [json.loads(line) for line in f]
df = pd.DataFrame(records)
print(f'Total records: {len(df)}')
agg = df.groupby('step')[['fg', 'rg']].agg(['mean', 'std']).round(4)
print(agg)
print()

# CRITICAL CHECK: are scores actually different across steps now?
fg_means = df.groupby('step')['fg'].mean().values
fg_var = float(np.var(fg_means))
print(f'Variance of FG means across steps: {fg_var:.6f}')
if fg_var < 1e-8:
    print('  🔴 Still bit-for-bit identical — fix did not work, or another bug')
else:
    print('  🟢 Real differences detected — proceed with phase-transition analysis')


In [ ]:
# Plot probe scores
agg_r = df.groupby('step').agg({'fg': ['mean', 'std'], 'rg': ['mean', 'std']}).reset_index()
steps = agg_r['step'].values
fg_mean = agg_r[('fg', 'mean')].values
fg_std = agg_r[('fg', 'std')].values
rg_mean = agg_r[('rg', 'mean')].values
rg_std = agg_r[('rg', 'std')].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].errorbar(steps, fg_mean, yerr=fg_std, marker='o', capsize=4, label='FabricationGuard')
axes[0].axhline(fg_mean[0], color='gray', linestyle='--', alpha=0.5, label='base (step 0)')
axes[0].set_xlabel('DPO step'); axes[0].set_ylabel('Mean FG score')
axes[0].set_title('FabricationGuard probe vs DPO step (forward-only)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].errorbar(steps, rg_mean, yerr=rg_std, marker='s', color='C1', capsize=4, label='ReasonGuard')
axes[1].axhline(rg_mean[0], color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('DPO step'); axes[1].set_ylabel('Mean RG score')
axes[1].set_title('ReasonGuard probe vs DPO step (forward-only)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / 'fig_probe_vs_step.png', dpi=150)
plt.show()


In [ ]:
# Fresh-probe AUROC: train binary classifier base-vs-final on activations, test intermediates
def load_acts(step_filter):
    acts = []
    for r in records:
        if r['step'] == step_filter:
            p = acts_dir / f"step{r['step']}_{r['pair_id']}.pt"
            if p.exists():
                acts.append(torch.load(p).float().numpy())
    return np.array(acts) if acts else None

X_base = load_acts(0)
final_step = max(df['step'].unique())
X_final = load_acts(final_step)

fresh_results = {}
if X_base is not None and X_final is not None and len(X_base) >= 5 and len(X_final) >= 5:
    X = np.vstack([X_base, X_final])
    y = np.concatenate([np.zeros(len(X_base)), np.ones(len(X_final))])
    clf = LogisticRegression(C=1.0, max_iter=2000)
    clf.fit(X, y)
    print(f'Fresh probe trained on {len(X_base)} base + {len(X_final)} final')
    for s in sorted(df['step'].unique()):
        Xs = load_acts(int(s))
        if Xs is not None and len(Xs) > 0:
            fresh_results[int(s)] = float(clf.predict_proba(Xs)[:, 1].mean())
    print('Fresh probe P(final-like) per step:')
    for s, p in fresh_results.items():
        print(f'  step {s}: {p:.3f}')


In [ ]:
# Phase transition heuristic: max delta between consecutive steps / avg delta
if fresh_results and len(fresh_results) >= 3:
    ks = sorted(fresh_results.keys())
    vals = [fresh_results[k] for k in ks]
    deltas = [abs(vals[i+1] - vals[i]) for i in range(len(vals)-1)]
    max_d = max(deltas)
    avg_d = sum(deltas) / len(deltas)
    ratio = max_d / max(avg_d, 1e-6)
    transition_step = ks[deltas.index(max_d) + 1]
    print(f'Max delta: {max_d:.4f} (transitioning at step {transition_step})')
    print(f'Avg delta: {avg_d:.4f}')
    print(f'Max/avg ratio: {ratio:.2f}')
    if ratio > CFG['gate_ratio']:
        print('  🔴 PHASE TRANSITION SIGNAL — grokking-like')
    elif ratio < 1.5:
        print('  🟢 GRADUAL — no grokking signature')
    else:
        print('  🟡 AMBIGUOUS')
    plt.figure(figsize=(7, 4.5))
    plt.plot(ks, vals, marker='D', color='C2', markersize=10)
    plt.axhline(0.5, color='gray', linestyle=':', label='chance')
    plt.xlabel('DPO step'); plt.ylabel('Fresh-probe P(final-like)')
    plt.title(f'Fresh probe progression (forward-only, ratio={ratio:.2f})')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(OUT / 'fig_fresh_probe.png', dpi=150)
    plt.show()
else:
    ratio = float('nan'); transition_step = None


## Phase 7 — FINAL_VERDICT + HF push


In [ ]:
verdict = {
    'experiment': 'nb41 grokking forward-only (post-fix)',
    'fix_applied': 'strip_language_model_from_lora_keys',
    'n_checkpoints': int(df['step'].nunique()),
    'n_holdout': CFG['n_holdout'],
    'fg_variance_across_steps': float(fg_var),
    'fix_worked': bool(fg_var > 1e-8),
    'probe_scores_per_step': {
        str(int(s)): {
            'fg_mean': float(df[df['step']==s]['fg'].mean()),
            'fg_std': float(df[df['step']==s]['fg'].std()),
            'rg_mean': float(df[df['step']==s]['rg'].mean()),
            'rg_std': float(df[df['step']==s]['rg'].std()),
            'n': int((df['step']==s).sum()),
        } for s in sorted(df['step'].unique())
    },
    'fresh_probe_per_step': {str(s): v for s, v in (fresh_results or {}).items()},
    'phase_transition_ratio': float(ratio) if ratio == ratio else None,
    'grokking_signal': bool(ratio > CFG['gate_ratio']) if ratio == ratio else None,
}
(OUT / 'FINAL_VERDICT.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))


In [ ]:
api = HfApi()
try: create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e: print(e)

readme = '''---
license: apache-2.0
tags: [grokking, dpo, probing, qwen36-27b]
---

# nb41 — Grokking forward-only (nb39 v2 with Qwen3.6 LoRA key fix)

Fixes the bug discovered in nb40: Qwen3.6-27B PEFT save creates keys with `.language_model.` infix; `PeftModel.from_pretrained` against dense reload silently fails (zero LoRA effect).

Uses forward-only methodology: feed `prompt + chosen` from nb37 pairs.json through each checkpoint, capture L31/L55 at end-of-think, score with FabricationGuard + ReasonGuard probes. ~10 min compute.

See FINAL_VERDICT.json for results.
'''
(OUT / 'README.md').write_text(readme)

api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                  repo_type='dataset', token=HF_TOKEN,
                  commit_message='nb41 grokking forward-only',
                  allow_patterns=['README.md', 'FINAL_VERDICT.json', 'fig_*.png', 'forward_only_results.jsonl'])
print(f'✓ pushed to https://huggingface.co/datasets/{CFG["output_repo"]}')


## Done

Read FINAL_VERDICT.json. Three outcomes:
- **`fix_worked: false`** → still bit-identical → deeper bug (escalate)
- **`grokking_signal: true`** (ratio > 2.0) → phase transition detected → paper material
- **`grokking_signal: false`** → gradual learning → preference learning is non-grokking-like in this regime; honest negative
